# 📊 Classical Baseline: Biomedical Link Prediction

This notebook trains and evaluates a **classical machine learning baseline** for drug-disease treatment prediction using the Hetionet knowledge graph.

## Pipeline Overview
1. Load sampled Hetionet `CtD` edges
2. Generate knowledge graph embeddings (TransE)
3. Prepare link prediction features
4. Train classical models (Logistic Regression, SVM)
5. Evaluate with biomedical-appropriate metrics (PR-AUC)
6. Save model for API integration

## Why Classical Baseline?
- Establishes performance floor for QML comparison
- Provides production-ready fallback
- Validates data pipeline correctness


In [ ]:
# Install dependencies (run once)
# !pip install pykeen scikit-learn pandas numpy matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report
)
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create directories
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)


## 1. Load Sampled KG Data


In [ ]:
# Load sampled CtD edges (from 01-kg-ingestion.ipynb)
df_edges = pd.read_csv("../data/hetionet_ctd_sample_500.csv")
print(f"Loaded {len(df_edges):,} CtD edges")

# Show sample
print("\nSample edges:")
print(df_edges.head())


## 2. Generate Knowledge Graph Embeddings

We'll use **TransE** embeddings from PyKEEN. For this PoC, we'll simulate embeddings since full training takes time.

> 💡 **Note**: In practice, run `kg_embedder.py` to generate real embeddings.


In [ ]:
def simulate_kg_embeddings(df_edges, embedding_dim=32, random_state=42):
    """Simulate KG embeddings for demonstration."""
    np.random.seed(random_state)
    
    # Get unique entities
    entities = pd.concat([df_edges["source"], df_edges["target"]]).unique()
    print(f"Generating embeddings for {len(entities)} entities")
    
    # Create random embeddings (in practice, use TransE from PyKEEN)
    embeddings = np.random.randn(len(entities), embedding_dim)
    
    # Create mapping
    entity_to_id = {entity: idx for idx, entity in enumerate(entities)}
    
    return embeddings, entity_to_id, entities

# Generate embeddings
embeddings, entity_to_id, entities = simulate_kg_embeddings(df_edges, embedding_dim=32)

# Save for consistency with kg_embedder.py
np.save("../data/embeddings.npy", embeddings)
pd.Series({idx: ent for ent, idx in entity_to_id.items()}).to_csv(
    "../data/id_to_entity.csv", index_label="id"
)
print("✅ Saved embeddings and mappings")


## 3. Prepare Link Prediction Features

For each (drug, disease) pair, create feature vector = [drug_emb; disease_emb]


In [ ]:
def prepare_link_features(df_edges, embeddings, entity_to_id, test_size=0.2, random_state=42):
    """Prepare features and labels for link prediction."""
    features = []
    labels = []
    valid_pairs = []
    
    for _, row in df_edges.iterrows():
        try:
            src_idx = entity_to_id[row["source"]]
            tgt_idx = entity_to_id[row["target"]]
            
            src_emb = embeddings[src_idx]
            tgt_emb = embeddings[tgt_idx]
            
            # Concatenate embeddings
            feature = np.concatenate([src_emb, tgt_emb])
            features.append(feature)
            labels.append(1)  # Positive samples
            valid_pairs.append((row["source"], row["target"]))
        except KeyError:
            continue
    
    X_pos = np.array(features)
    y_pos = np.array(labels)
    
    print(f"Prepared {len(X_pos):,} positive samples")
    
    # Generate negative samples (1:1 ratio)
    np.random.seed(random_state)
    unique_sources = df_edges["source"].unique()
    unique_targets = df_edges["target"].unique()
    
    X_neg = []
    y_neg = []
    existing_pairs = set(zip(df_edges["source"], df_edges["target"]))
    
    attempts = 0
    while len(X_neg) < len(X_pos) and attempts < len(X_pos) * 10:
        src = np.random.choice(unique_sources)
        tgt = np.random.choice(unique_targets)
        
        if (src, tgt) not in existing_pairs:
            try:
                src_idx = entity_to_id[src]
                tgt_idx = entity_to_id[tgt]
                src_emb = embeddings[src_idx]
                tgt_emb = embeddings[tgt_idx]
                feature = np.concatenate([src_emb, tgt_emb])
                X_neg.append(feature)
                y_neg.append(0)
                existing_pairs.add((src, tgt))  # Avoid duplicates
            except KeyError:
                pass
        attempts += 1
    
    X_neg = np.array(X_neg)
    y_neg = np.array(y_neg)
    
    print(f"Generated {len(X_neg):,} negative samples")
    
    # Combine and split
    X = np.vstack([X_pos, X_neg])
    y = np.hstack([y_pos, y_neg])
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    print(f"Train set: {len(X_train):,} samples ({y_train.sum():,} positive)")
    print(f"Test set: {len(X_test):,} samples ({y_test.sum():,} positive)")
    
    return X_train, X_test, y_train, y_test

# Prepare features
X_train, X_test, y_train, y_test = prepare_link_features(
    df_edges, embeddings, entity_to_id
)


## 4. Reduce Dimensionality for QML Compatibility

Reduce from 64D (32+32) to 5D for quantum encoding


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

def reduce_dimensions_for_qml(X_train, X_test, target_dim=5, random_state=42):
    """Reduce feature dimensionality for QML compatibility."""
    print(f"Reducing from {X_train.shape[1]}D to {target_dim}D")
    
    pca = PCA(n_components=target_dim, random_state=random_state)
    X_train_reduced = pca.fit_transform(X_train)
    X_test_reduced = pca.transform(X_test)
    
    # Normalize for amplitude encoding
    X_train_reduced = normalize(X_train_reduced, norm='l2')
    X_test_reduced = normalize(X_test_reduced, norm='l2')
    
    # Save reduced embeddings (mimics kg_embedder.py output)
    np.save("../data/embeddings_qml_5d.npy", np.vstack([X_train_reduced, X_test_reduced]))
    
    print(f"Reduced train shape: {X_train_reduced.shape}")
    print(f"Reduced test shape: {X_test_reduced.shape}")
    
    return X_train_reduced, X_test_reduced, pca

X_train_5d, X_test_5d, pca = reduce_dimensions_for_qml(X_train, X_test, target_dim=5)


## 5. Train Classical Models


In [ ]:
def train_and_evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Train and evaluate a single model."""
    print(f"\n{'='*50}")
    print(f"Training {model_name}")
    print(f"{'='*50}")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Metrics
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'num_parameters': model.coef_.size if hasattr(model, 'coef_') else -1
    }
    
    # Print results
    for metric, value in metrics.items():
        if metric != 'num_parameters':
            print(f"{metric.replace('_', ' ').title()}: {value:.4f}")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    return model, scaler, metrics

# Train Logistic Regression
lr_model, lr_scaler, lr_metrics = train_and_evaluate_model(
    LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    X_train_5d, X_test_5d, y_train, y_test, "Logistic Regression"
)

# Train SVM
svm_model, svm_scaler, svm_metrics = train_and_evaluate_model(
    SVC(random_state=42, probability=True, class_weight='balanced', kernel='rbf'),
    X_train_5d, X_test_5d, y_train, y_test, "SVM"
)


## 6. Compare Models and Save Best


In [ ]:
# Compare PR-AUC (most important for imbalanced biomedical data)
print(f"\n{'='*50}")
print("Model Comparison (PR-AUC)")
print(f"{'='*50}")
print(f"Logistic Regression: {lr_metrics['pr_auc']:.4f}")
print(f"SVM: {svm_metrics['pr_auc']:.4f}")

# Select best model
if lr_metrics['pr_auc'] >= svm_metrics['pr_auc']:
    best_model = lr_model
    best_scaler = lr_scaler
    best_metrics = lr_metrics
    best_name = "LogisticRegression"
else:
    best_model = svm_model
    best_scaler = svm_scaler
    best_metrics = svm_metrics
    best_name = "SVM"

print(f"\n✅ Best model: {best_name}")


## 7. Save Model for API Integration


In [ ]:
# Save best model and scaler
joblib.dump(best_model, f"../models/classical_{best_name.lower()}.joblib")
joblib.dump(best_scaler, "../models/scaler.joblib")
print(f"✅ Saved {best_name} model and scaler")

# Save metrics for dashboard
results_df = pd.DataFrame([{
    'classical_accuracy': best_metrics['accuracy'],
    'classical_precision': best_metrics['precision'],
    'classical_recall': best_metrics['recall'],
    'classical_f1': best_metrics['f1'],
    'classical_roc_auc': best_metrics['roc_auc'],
    'classical_pr_auc': best_metrics['pr_auc'],
    'classical_num_parameters': best_metrics['num_parameters']
}])

results_df.to_csv("../results/classical_baseline.csv", index=False)
print("✅ Saved classical baseline metrics")


## 🎯 Next Steps

1. **Run `classical_baseline/train_baseline.py`** to verify programmatic training
2. **Proceed to `03-qml-training.ipynb`** for quantum model training
3. **Test predictions** with known drug-disease pairs:
   - Load model: `joblib.load('../models/classical_logisticregression.joblib')`
   - Predict: `model.predict_proba(scaler.transform([feature_vector]))`
